# Open, encode, decode, and visualize a triangle-mesh sequence

This uses the real `4d_files/Rafa_Approves_hd_4k` OBJ sequence and attempts every codec registered in the current environment. Optional source-only and native-process codecs report missing prerequisites instead of disappearing from the results. The current canonical value is a finite `TriangleMesh` sequence; point clouds, volumes, Gaussian splats, and live streams are not part of this API slice.

In [ ]:
import os
import time
from pathlib import Path
import numpy as np

from open4d import MemoryFrameProvider, Sequence
from open4d.codec import available_codecs, decode_sequence, encode_sequence
from open4d.io import inspect_sequence, open_sequence
from open4d.visualization import visualize

ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "4d_files/Rafa_Approves_hd_4k").is_dir()
)
DATASET = ROOT / "4d_files/Rafa_Approves_hd_4k"
ARTIFACT_DIR = Path(os.environ.get("OPEN4D_ARTIFACT_DIR", ROOT / ".context/rafa_codecs"))
DEMO_FRAMES = int(os.environ.get("OPEN4D_DEMO_FRAMES", "2")) or None
DEVICE = os.environ.get("OPEN4D_NOTEBOOK_DEVICE", "cpu")
REQUIRE_ALL = os.environ.get("OPEN4D_NOTEBOOK_REQUIRE_ALL") == "1"
CODEC_INFOS = {info.id: info for info in available_codecs()}
CODECS = tuple(CODEC_INFOS)
ENCODE_OPTIONS = {
    "klt": dict(resolution=15, num_components=8, block_size=4, k_total=256, training_frames=(0,)),
    "n4mc": dict(resolution=15, epochs=20, hidden_channels=(8, 16), latent_channels=8, device=DEVICE),
    "qndf": dict(coarse_size=100, num_subdiv=0, epochs=20, hidden_dim=8, num_layers=3, batch_size=256, device=DEVICE),
    "qndf-int8": dict(coarse_size=100, num_subdiv=0, epochs=20, hidden_dim=8, num_layers=3, batch_size=256, device=DEVICE),
    "temporal-delta": dict(face_budget=100, quantization_bits=12),
    "temporal-pca": dict(face_budget=100, quantization_bits=12, components=3),
}
DECODE_OPTIONS = {codec: {"device": DEVICE} for codec in ("klt", "n4mc", "qndf")}
DECODE_OPTIONS["qndf-int8"] = {"device": "cpu"}

def options_for(codec):
    encode = dict(ENCODE_OPTIONS.get(codec, {}))
    decode = dict(DECODE_OPTIONS.get(codec, {}))
    if codec in {"vdmc", "faster_vdmc"}:
        prefix = f"OPEN4D_{codec.upper()}"
        names = (f"{prefix}_ENCODER", f"{prefix}_DECODER", f"{prefix}_ENCODER_CONFIG", f"{prefix}_DECODER_CONFIG")
        missing = [name for name in names if not os.environ.get(name)]
        if missing:
            raise RuntimeError(f"set {', '.join(missing)} for the native V-DMC adapter")
        encode.update(encoder=os.environ[names[0]], encoder_config=os.environ[names[2]], decoder_config=os.environ[names[3]])
        decode.update(decoder=os.environ[names[1]])
    return encode, decode

In [ ]:
info = inspect_sequence(DATASET)
sequence = open_sequence(DATASET, fps=30)
selected = sequence if DEMO_FRAMES is None else sequence[:DEMO_FRAMES]
demo = Sequence(MemoryFrameProvider(
    tuple(selected), metadata=selected.metadata, topology=selected.topology,
    has_constant_vertex_count=selected.has_constant_vertex_count,
    has_vertex_correspondence=selected.has_vertex_correspondence,
))
print(f"Loaded {info.frame_count} {info.format.upper()} frames; using {len(demo)} for this run.")

In [ ]:
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
results = []
decoded_sequences = {}
for codec in CODECS:
    codec_info = CODEC_INFOS[codec]
    artifact = ARTIFACT_DIR / f"rafa-{codec}{codec_info.suffixes[0]}"
    try:
        encode_options, decode_options = options_for(codec)
        start = time.perf_counter()
        encode_sequence(demo, artifact, codec=codec, overwrite=True, **encode_options)
        encode_s = time.perf_counter() - start
        start = time.perf_counter()
        candidate = decode_sequence(artifact, **decode_options)
        exact = True
        assert candidate.metadata == demo.metadata and len(candidate) == len(demo)
        for expected, actual in zip(demo, candidate, strict=True):
            assert (actual.frame_index, actual.timestamp, actual.metadata) == (expected.frame_index, expected.timestamp, expected.metadata)
            exact &= np.array_equal(actual.geometry.positions, expected.geometry.positions)
            exact &= np.array_equal(actual.geometry.triangles, expected.geometry.triangles)
            assert len(actual.geometry.positions) and len(actual.geometry.triangles)
        if codec_info.lossless:
            assert exact
        decode_s = time.perf_counter() - start
    except Exception as error:
        results.append(dict(codec=codec, status="error", detail=f"{type(error).__name__}: {error}"))
    else:
        results.append(dict(codec=codec, status="ok", size=artifact.stat().st_size, encode_s=encode_s, decode_s=decode_s, exact=exact))
        decoded_sequences[codec] = candidate

In [ ]:
print("codec                 status       size (MB)  encode (s)  decode+verify (s)")
for row in results:
    if row["status"] == "ok":
        print(f"{row['codec']:<21} ok           {row['size'] / 1_000_000:>9.2f}  {row['encode_s']:>10.3f}  {row['decode_s']:>17.3f}")
    else:
        print(f"{row['codec']:<21} error        {row['detail']}")
print(f"Attempted all {len(CODECS)} registered codecs; {len(decoded_sequences)} decoded successfully.")
failed = [row['codec'] for row in results if row['status'] != 'ok']
if REQUIRE_ALL and failed:
    raise RuntimeError(f"registered codecs failed: {', '.join(failed)}")

In [ ]:
try:
    if os.environ.get("OPEN4D_NOTEBOOK_HEADLESS") == "1":
        print("Headless run: visualization calls skipped.")
    else:
        for codec, decoded in decoded_sequences.items():
            print(f"Visualizing {codec}; close its viewer to continue.")
            visualize(decoded, up="y", fps=30)
finally:
    for decoded in decoded_sequences.values():
        decoded.close()
    sequence.close()